In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/Users/prady/Code/RegimeFactorZoo")
os.chdir(PROJECT_ROOT)

print("Working directory:", Path.cwd())

Working directory: /Users/prady/Code/RegimeFactorZoo


# Notebook 05 — Path A Phase 1: Data Construction

**Goal**: Transform raw CRSP/Compustat into the (permno, year) panel
with ME, BE, and B/M needed for FF3 portfolio formation.

**Methodology references**:
- Fama & French (1993), Section II (pp. 10–17) — universe filters, sort design
- Davis, Fama & French (2000) — book equity formula refinement

**Pre-conditions**:
- WRDS pulls complete: data/pulls/crsp_monthly.parquet (3.4M rows),
  data/pulls/compustat_annual.parquet (572K rows),
  data/pulls/crsp_compustat_link.parquet (33K rows)
- Kernel: Python 3 (RFZ venv)

In [2]:
"""Write yourself: imports (pandas, numpy), set pd.options.display.max_columns = None, define a DATA = Path("../data/pulls") constant."""

import pandas as pd
import numpy as np
from pathlib import Path
pd.options.display.max_columns = None
Data = Path("data/pulls")

## Section 1 — Inspect raw CRSP

Goal: load crsp_monthly, verify filters held, examine schema, count NA
in the columns we'll use downstream.

In [3]:
"""What it should do (no code from me):

Load crsp_monthly.parquet
Print shape, columns, dtypes, date range
Assert shrcd ∈ {10, 11} and exchcd ∈ {1, 2, 3} (both should pass since the SQL filtered)
Print NA counts for prc, shrout, ret
If any assertion fails, the SQL filter leaked. If NA counts are high in ret, that's normal for IPO/delisting months — handle later.
"""

crsp_month = pd.read_parquet(Data / "crsp_monthly.parquet")

print("Confirming data load:\n",crsp_month.head(),"\n")

print("The Shape of crsp is: \n",crsp_month.shape,"\n")

print("The Columns of crsp are: \n",crsp_month.columns,"\n")

print("The dtypes of crsp are:\n",crsp_month.dtypes,"\n")

min_date = crsp_month["date"].min()
max_date = crsp_month["date"].max()
print("The range of crsp is:\n",max_date - min_date,"\n")

assert crsp_month["shrcd"].isin([10, 11]).all(), "found values outside {10, 11} ~ shrcd leaked"
assert crsp_month["exchcd"].isin([1, 2, 3]).all(), "found values other than {1,2,3} ~ exchcd leaked"

na_count_prc = crsp_month["prc"].isna().sum()
na_count_shrout = crsp_month["shrout"].isna().sum()
na_count_ret = crsp_month["ret"].isna().sum()

print("Total NA rows in prc, shrout, ret columns are -\n",na_count_prc, "\n", na_count_shrout, "\n", na_count_ret, "\n")



Confirming data load:
    permno  permco       date       ret      retx  shrout      prc     vol  \
0   10000    7952 1986-01-31      <NA>      <NA>  3680.0   -4.375  1771.0   
1   10000    7952 1986-02-28 -0.257143 -0.257143  3680.0    -3.25   828.0   
2   10000    7952 1986-03-31  0.365385  0.365385  3680.0  -4.4375  1078.0   
3   10000    7952 1986-04-30 -0.098592 -0.098592  3793.0     -4.0   957.0   
4   10000    7952 1986-05-30 -0.222656 -0.222656  3793.0 -3.10938  1074.0   

   cfacshr  shrcd  exchcd  siccd ticker  
0      1.0     10       3   3990  OMFGA  
1      1.0     10       3   3990  OMFGA  
2      1.0     10       3   3990  OMFGA  
3      1.0     10       3   3990  OMFGA  
4      1.0     10       3   3990  OMFGA   

The Shape of crsp is: 
 (3409860, 13) 

The Columns of crsp are: 
 Index(['permno', 'permco', 'date', 'ret', 'retx', 'shrout', 'prc', 'vol',
       'cfacshr', 'shrcd', 'exchcd', 'siccd', 'ticker'],
      dtype='object') 

The dtypes of crsp are:
 permno       

## Section 2 — Compute market equity (ME)

Formula (Lesson 2): ME = |prc| × shrout / 1000, units = $ millions.

Sanity targets (Dec 31, 2023, with all U.S. universe filtered to common
shares on major exchanges):
- Apple (permno 14593) ME should be roughly $2.9–3.0T
  
- Microsoft (permno 10107) should be roughly $2.7–2.8T
- Top 10 by ME should be the names you'd guess

In [4]:
"""ME computation (YOURS to write)
This is the cell you write next. Constraints:

Add me column to the CRSP DataFrame
Handle the negative-price convention (Lesson 2, Q2)
Verify Apple Dec 2023 lands in the expected range
Verify top 10 at end of 2023 are recognizable
Drop rows where me cannot be computed (price OR shares missing)
When you have a draft, paste it back and I'll tell you what's right, what's subtle, what's missing.

"""
crsp_month["absprc"] = crsp_month["prc"].abs()    #negative to absolute value of the prc column

crsp_month["me"] = crsp_month["absprc"] * crsp_month["shrout"] / 1_000

crsp_month = crsp_month.sort_values(by=['permno', 'date']).reset_index(drop=True)

crsp_month = crsp_month.dropna(subset=['me']).reset_index(drop=True)

print(crsp_month["me"].head())


#check

apple_dec2023 = crsp_month[(crsp_month['date'] == '2023-12-29') & (crsp_month['permno'] == 14593)]
print(apple_dec2023[['permno', 'date', 'absprc', 'shrout', 'me']])


top10_2023 = crsp_month[crsp_month['date'] == '2023-12-29']

check = top10_2023.nlargest(10, 'me')

print(check[['permno', 'absprc', 'shrout', 'me']]) 

0         16.1
1        11.96
2        16.33
3       15.172
4    11.793878
Name: me, dtype: Float64
        permno       date  absprc      shrout             me
380358   14593 2023-12-29  192.53  15460223.0  2976556.73419
         permno     absprc      shrout              me
380358    14593     192.53  15460223.0   2976556.73419
14212     10107  376.04001   7432262.0  2794827.876803
2879795   84788     151.94  10242000.0      1556169.48
3000933   86580     495.22   2470000.0       1223193.4
3232474   90319     139.69   5899000.0       824031.31
376951    14542  140.92999   5691000.0    802032.57309
3393585   93436     248.48   3185000.0        791408.8
316550    13407  353.95999   2211000.0    782605.53789
1579690   50876  582.91998    949307.0   553370.017454
3376251   93002    1116.25    468141.0    522562.39125


In [5]:
#SECTION 3
#Inspect raw Compustat

#Goal: load crsp_monthly, verify filters held, examine schema, count NA
#in the columns we'll use downstream.

In [6]:

compustat_annual = pd.read_parquet(Data / "compustat_annual.parquet")

print("Confirming data load:\n",compustat_annual.head(),"\n")

print("The Shape of cmpstat is: \n",compustat_annual.shape,"\n")

print("The Columns of cmpstat are: \n",compustat_annual.columns,"\n")

print("The dtypes of cmpstat are:\n",compustat_annual.dtypes,"\n")
compustat_annual["datadate"] = pd.to_datetime(compustat_annual["datadate"])
min_date = compustat_annual["datadate"].min()
max_date = compustat_annual["datadate"].max()
print("The range of compustat is:\n",max_date - min_date,"\n")

start_rows = len(compustat_annual)
print("Start rows:", start_rows)

assert start_rows == 572_820, f"Unexpected start rows: {start_rows:,}"

comp = compustat_annual[
    (compustat_annual["indfmt"] == "INDL") &
    (compustat_annual["datafmt"] == "STD") &
    (compustat_annual["popsrc"] == "D") &
    (compustat_annual["consol"] == "C")
].copy()

end_rows = len(comp)
rows_dropped = start_rows - end_rows

print("Rows after 4 format filters:", end_rows)
print("Rows dropped by 4 format filters:", rows_dropped, "\n")


max_group_size = comp.groupby(["gvkey", "fyear"]).size().max()

print("Max rows per (gvkey, fyear):", max_group_size)

comp = comp.sort_values(["gvkey", "fyear", "datadate"])
comp = comp.drop_duplicates(subset=["gvkey", "fyear"], keep="last").copy()

max_group_size = comp.groupby(["gvkey", "fyear"]).size().max()

print("Max rows per (gvkey, fyear):", max_group_size)

assert max_group_size == 1, "Duplicate (gvkey, fyear) rows remain"

Confirming data load:
     gvkey   datadate  fyear  fyr indfmt consol popsrc datafmt     at     lt  \
0  001000 1962-12-31   1962   12   INDL      C      D     STD   <NA>   <NA>   
1  001000 1963-12-31   1963   12   INDL      C      D     STD   <NA>  0.345   
2  001000 1964-12-31   1964   12   INDL      C      D     STD  1.416  0.809   
3  001000 1965-12-31   1965   12   INDL      C      D     STD   2.31  1.818   
4  001000 1966-12-31   1966   12   INDL      C      D     STD   2.43  1.596   

     ceq    seq  txditc  pstkrv  pstkl  pstk   sale   cogs   xsga     ib  \
0  0.552   <NA>    <NA>    <NA>    0.0   0.0    1.6   <NA>   <NA>   0.12   
1  0.553  0.553   0.008     0.0    0.0   0.0  1.457  1.065  0.346  0.003   
2  0.607  0.607    0.02     0.0    0.0   0.0  2.032  1.474  0.431  0.039   
3  0.491  0.491     0.0     0.0    0.0   0.0  1.688  1.342  0.506 -0.197   
4  0.834  0.834     0.0     0.0    0.0   0.0  4.032  2.828  0.679  0.201   

      ni  oibdp     dp   capx  ppent  ppegt  

In [7]:
#Section 4: Book Equity formula 

#two fallback ladders
#"Book equity" = what's left for shareholders on paper after subtracting what the company owes. 
#Sounds like one number. It isn't — Compustat reports it inconsistently across companies, so you need a fallback ladder: 
#try the best source, fall back to the next if it's missing. 

#This is the Davis-Fama-French (2000) formula.

# Stockholders' equity (SE): try seq → else ceq + pstk → else al − lt

# Preferred stock (PS): try pstkrv → else pstkl → else pstk → else 0

# BE = SE + txditc − PS

# BE comes out negative = drop

#Write a function book_equity(row) implementing both ladders, apply it to comp, then:

# count negative-BE rows, drop them
# sanity check: print BE for a known 
# firm (e.g. Apple gvkey 001690) for a recent year — should be tens of billions (Apple ~$60-80B book equity)


In [8]:
def book_equity(row):
    # Stockholders equity
    if pd.notna(row["seq"]):
        se = row["seq"]
    elif pd.notna(row["ceq"]) and pd.notna(row["pstk"]):
        se = row["ceq"] + row["pstk"]
    elif pd.notna(row["at"]) and pd.notna(row["lt"]):
        se = row["at"] - row["lt"]
    else:
        se = np.nan

    # Preferred stock
    if pd.notna(row["pstkrv"]):
        ps = row["pstkrv"]
    elif pd.notna(row["pstkl"]):
        ps = row["pstkl"]
    elif pd.notna(row["pstk"]):
        ps = row["pstk"]
    else:
        ps = 0

    # Deferred taxes
    txditc = row["txditc"] if pd.notna(row["txditc"]) else 0

    return se + txditc - ps


#BE formula
comp["be"] = comp.apply(book_equity, axis=1)

#drop negative BE
negative_be_count = (comp["be"] < 0).sum()
print("Negative BE rows:", negative_be_count)

comp = comp[comp["be"] > 0].copy()

#Apple BE check
apple_be = comp[(comp["gvkey"] == "001690")].sort_values("fyear").tail(5)[["gvkey", "fyear", "datadate", "be"]]

print("\nApple BE sanity check:")
print(apple_be)

Negative BE rows: 48850

Apple BE sanity check:
        gvkey  fyear   datadate       be
14042  001690   2020 2020-09-30  68139.0
14043  001690   2021 2021-09-30  63090.0
14044  001690   2022 2022-09-30  51510.0
14045  001690   2023 2023-09-30  62747.0
14046  001690   2024 2024-09-30  57247.0
